# تحليل أداء طلبات الوجبات

يشغّل هذا الدفتر التحليل من ملفات CSV التي يضيفها المستخدم محلياً إلى `data/raw/`. لا يحتوي المستودع العام على بيانات المصدر، ولا يكتب الدفتر إليها. يجب استخدام `train.csv` للتحليل، وملفات المركز والوجبة للوصف؛ ملف `test.csv` بلا هدف `num_orders`.

النتائج وصفية ورصدية. اختبارات Mann–Whitney وSpearman لا تثبت السببية.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'data' / 'raw' / 'train.csv').exists():
            return path
    raise FileNotFoundError('Place the source CSV files in data/raw/ before running the notebook.')

ROOT = find_root()
RAW = ROOT / 'data' / 'raw'
FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

def audit(frame, label):
    total = frame['num_orders'].sum() if 'num_orders' in frame else frame['total_orders'].sum() if 'total_orders' in frame else None
    suffix = f' | orders={total:,.0f}' if total is not None else ''
    print(f'{label}: rows={len(frame):,}, columns={frame.shape[1]}{suffix}')

train = pd.read_csv(RAW / 'train.csv')
centers = pd.read_csv(RAW / 'fulfilment_center_info.csv')
meals = pd.read_csv(RAW / 'meal_info.csv')
for name, frame in [('train', train), ('centers', centers), ('meals', meals)]:
    audit(frame, name)
    print('columns:', frame.columns.tolist(), '| missing:', int(frame.isna().sum().sum()))
print('duplicate train keys:', int(train.duplicated(['week', 'center_id', 'meal_id']).sum()))
print('duplicate center keys:', int(centers['center_id'].duplicated().sum()))
print('duplicate meal keys:', int(meals['meal_id'].duplicated().sum()))

## الربط وفحص الجودة
تُطبع الأبعاد ومجاميع الطلبات قبل كل ربط وبعده، وتُفحص المفاتيح غير المطابقة. لا تُحذف السجلات المتطرفة تلقائياً.

In [ ]:
orders = train.copy()
audit(orders, 'before center join')
orders = orders.merge(centers, on='center_id', how='left', validate='many_to_one', indicator='_center')
audit(orders, 'after center join')
print('unmatched centers:', int(orders['_center'].eq('left_only').sum()))
orders = orders.drop(columns='_center')
audit(orders, 'after dropping center diagnostic')
audit(orders, 'before meal join')
orders = orders.merge(meals, on='meal_id', how='left', validate='many_to_one', indicator='_meal')
audit(orders, 'after meal join')
print('unmatched meals:', int(orders['_meal'].eq('left_only').sum()))
orders = orders.drop(columns='_meal')
audit(orders, 'after dropping meal diagnostic')
assert len(orders) == len(train) and orders['num_orders'].sum() == train['num_orders'].sum()
print('missing values:', int(orders.isna().sum().sum()))
print('duplicate business keys:', int(orders.duplicated(['week', 'center_id', 'meal_id']).sum()))
print('checkout price above base price:', int(orders['checkout_price'].gt(orders['base_price']).sum()))
q1, q3 = orders['num_orders'].quantile([.25, .75]); iqr = q3 - q1
outliers = orders['num_orders'].lt(q1 - 1.5 * iqr) | orders['num_orders'].gt(q3 + 1.5 * iqr)
print('IQR-flagged rows retained for review:', int(outliers.sum()))
weeks = set(range(int(orders['week'].min()), int(orders['week'].max()) + 1))
gaps = orders.groupby(['center_id', 'meal_id'])['week'].agg(lambda values: len(weeks - set(values)))
print('series with missing weeks:', int(gaps.gt(0).sum()), '| absent center-meal weeks:', int(gaps.sum()))
analysis = orders.copy()
audit(analysis, 'analysis after documented no-drop decision')

## مؤشرات الطلب والتذبذب
الإيراد تقديري (`checkout_price × num_orders`) وليس ربحاً. التذبذب CV محسوب لكل سلسلة مركز-وجبة على الأسابيع المسجلة فقط.

In [ ]:
analysis = analysis.assign(estimated_revenue=analysis['checkout_price'] * analysis['num_orders'], discount_pct=np.where(analysis['base_price'].gt(0), (analysis['base_price'] - analysis['checkout_price']) / analysis['base_price'] * 100, np.nan))
weekly = analysis.groupby('week', as_index=False)['num_orders'].sum()
categories = analysis.groupby('category', as_index=False)['num_orders'].sum().sort_values('num_orders', ascending=False)
meals_demand = analysis.groupby(['meal_id', 'category', 'cuisine'], as_index=False)['num_orders'].sum().sort_values('num_orders', ascending=False)
meals_demand['cumulative_share'] = meals_demand['num_orders'].cumsum() / meals_demand['num_orders'].sum()
top_80 = meals_demand.loc[meals_demand['cumulative_share'].shift(fill_value=0) < .8]
center_week = analysis.groupby(['center_type', 'center_id', 'week'], as_index=False)['num_orders'].sum()
center_means = center_week.groupby(['center_type', 'center_id'], as_index=False)['num_orders'].mean().rename(columns={'num_orders':'mean_weekly_orders'})
center_summary = center_means.groupby('center_type', as_index=False).agg(mean_weekly_orders_per_center=('mean_weekly_orders','mean'), centers=('center_id','nunique'))
audit(center_summary, 'center summary before totals join')
center_totals = analysis.groupby('center_type', as_index=False)['num_orders'].sum().rename(columns={'num_orders':'total_orders'})
center_summary = center_summary.merge(center_totals, on='center_type', validate='one_to_one')
audit(center_summary, 'center summary after totals join')
assert center_summary['total_orders'].sum() == analysis['num_orders'].sum()
meal_weekly = analysis.groupby(['center_id','meal_id','week'], as_index=False)['num_orders'].sum()
variability = meal_weekly.groupby(['center_id','meal_id'])['num_orders'].agg(mean='mean', std='std', series_total_orders='sum').reset_index()
variability['cv'] = variability['std'] / variability['mean'].replace(0, np.nan)
print('total orders:', f"{analysis['num_orders'].sum():,.0f}")
print('mean weekly orders:', f"{weekly['num_orders'].mean():,.2f}")
print('estimated revenue:', f"{analysis['estimated_revenue'].sum():,.2f}")
print('promotion row shares:', analysis[['emailer_for_promotion','homepage_featured']].mean().to_dict())
display(categories, top_80, center_summary, variability.sort_values('cv', ascending=False).head(15))

## الاختبارات الإحصائية
المقارنة رصدية ومجموعات الاختبار المتعددة غير مصححة؛ لا تستنتج السببية من قيمة p.

In [ ]:
from scipy.stats import mannwhitneyu, spearmanr
results = []
for flag in ['emailer_for_promotion', 'homepage_featured']:
    for (center_id, meal_id), group in analysis.groupby(['center_id','meal_id']):
        no = group.loc[group[flag].eq(0), 'num_orders']; yes = group.loc[group[flag].eq(1), 'num_orders']
        if len(no) >= 2 and len(yes) >= 2:
            test = mannwhitneyu(yes, no, alternative='two-sided')
            baseline = no.median()
            results.append({'flag':flag,'center_id':center_id,'meal_id':meal_id,'n_yes':len(yes),'n_no':len(no),'median_change_pct':(yes.median()-baseline)/baseline*100 if baseline else np.nan,'p_value':test.pvalue})
promotion_tests = pd.DataFrame(results)
promotion_summary = promotion_tests.groupby('flag').agg(strata=('p_value','size'), median_within_stratum_change_pct=('median_change_pct','median'), unadjusted_p_below_005=('p_value',lambda x:int(x.lt(.05).sum())))
display(promotion_summary)
spearman_data = analysis[['discount_pct','num_orders']].replace([np.inf,-np.inf],np.nan)
audit(spearman_data, 'Spearman data before dropping nulls')
spearman_data = spearman_data.dropna()
audit(spearman_data, 'Spearman data after dropping nulls')
spearman_result = spearmanr(spearman_data['discount_pct'],spearman_data['num_orders'])
print('Spearman rho:', round(spearman_result.statistic,4), '| p:', spearman_result.pvalue, '| n:',len(spearman_data))

## الرسوم
تُحفظ الرسوم في `reports/figures/` خارج `data/raw/`.

In [ ]:
fig, ax = plt.subplots(figsize=(10,4)); ax.plot(weekly['week'],weekly['num_orders']); ax.set(title='الطلب الأسبوعي المسجل',xlabel='الأسبوع',ylabel='الطلبات'); fig.tight_layout(); fig.savefig(FIGURES/'weekly_demand.svg'); plt.close(fig)
fig, ax = plt.subplots(figsize=(10,5)); ax.bar(categories['category'].astype(str),categories['num_orders']); ax.tick_params(axis='x',rotation=45); ax.set(title='الطلبات المسجلة حسب الفئة',xlabel='الفئة',ylabel='الطلبات'); fig.tight_layout(); fig.savefig(FIGURES/'category_demand.svg'); plt.close(fig)
for flag,title,filename in [('emailer_for_promotion','البريد الترويجي','email_promotion_orders_boxplot.svg'),('homepage_featured','الظهور في الصفحة الرئيسية','homepage_featured_orders_boxplot.svg')]:
    fig,ax=plt.subplots(figsize=(7,4)); ax.boxplot([analysis.loc[analysis[flag].eq(v),'num_orders'] for v in [0,1]],showfliers=False); ax.set(title=f'توزيع الطلب حسب {title}',ylabel='الطلبات'); fig.tight_layout(); fig.savefig(FIGURES/filename); plt.close(fig)
sample=analysis.sample(min(800,len(analysis)),random_state=42)
fig,ax=plt.subplots(figsize=(7,4)); ax.scatter(sample['discount_pct'],sample['num_orders'],s=10,alpha=.3,rasterized=True); ax.set(title='الخصم والطلبات المسجلة',xlabel='الخصم (%)',ylabel='الطلبات'); fig.tight_layout(); fig.savefig(FIGURES/'discount_vs_orders.svg'); plt.close(fig)
sample_var=variability.sample(min(500,len(variability)),random_state=42)
fig,ax=plt.subplots(figsize=(7,4)); ax.scatter(sample_var['series_total_orders'],sample_var['cv'],s=16,alpha=.5,rasterized=True); ax.set(title='حجم الطلب وتذبذبه حسب المركز والوجبة',xlabel='طلبات السلسلة',ylabel='CV'); fig.tight_layout(); fig.savefig(FIGURES/'demand_volume_vs_variability.svg'); plt.close(fig)
print('Saved figures:', sorted(p.name for p in FIGURES.glob('*.svg')))

## حدود التحليل

- البيانات رصدية؛ اختبارات Mann–Whitney وSpearman لا تثبت السببية.
- الاختبارات المتعددة غير مصححة والمشاهدات الأسبوعية المتكررة قد لا تكون مستقلة.
- الأسابيع الغائبة لم تُملأ بأصفار، ولا تتضمن البيانات المخزون الفعلي أو الهدر أو التكاليف.
- تقدير الإيراد ليس ربحاً، ولا توضح الملفات العملة؛ المدن مشفرة والأسابيع ليست تواريخ تقويمية.
